# Lesson 06 - Multi-agent customer-support handoffs

**Level:** advanced | **Demo time:** 15 minutes | **Builds on:** Lessons 02 and 04

A handoff workflow uses a mesh of specialized agents. Each participant can answer, use its own tools, or transfer control to another participant. This is useful when a conversation crosses domains and the specialist should directly own its part of the interaction.

## Learning objectives

- Give each agent a narrow responsibility and minimum tool set.
- Define allowed handoff paths explicitly.
- Run a reproducible human-in-the-loop conversation.
- Recognize when a deterministic workflow is preferable to agent-directed routing.

## Workshop topology

```text
                 +------------------+
Customer ------> |  Triage agent    |
                 +---+----------+---+
                     |          |
              +------v--+   +---v--------+
              | Orders  |   | Returns    |
              +---------+   +-----+------+
                                  |
                            +-----v------+
                            | Refunds    |
                            +------------+
```

Descriptions are routing metadata. Instructions control specialist behavior. Tools enforce capability boundaries.

In [ ]:
from agent_framework import Agent, AgentResponse, Content, Message, WorkflowAgent
from agent_framework.orchestrations import HandoffAgentUserRequest, HandoffBuilder

from maf_foundry_hackathon.config import (
    WorkshopSettings,
    create_chat_client,
    create_credential,
)
from maf_foundry_hackathon.support_tools import (
    create_refund_request,
    create_return_request,
    lookup_order,
    search_support_policy,
)

settings = WorkshopSettings.from_env()
credential = create_credential()
chat_client = create_chat_client(settings, credential)

## Create specialized participants

All agents can share one thread-safe client on the same event loop, while their definitions and tools remain separate. `require_per_service_call_history_persistence=True` keeps the handoff conversation available across each service call.

In [ ]:
triage_agent = Agent(
    client=chat_client,
    name="triage_agent",
    description="Frontline agent that identifies the customer's support need and routes it.",
    instructions=(
        "You are frontline support triage. Ask for missing essentials, then hand off "
        "order tracking, returns, or refunds to the matching specialist. Do not perform specialist work."
    ),
    require_per_service_call_history_persistence=True,
)

order_agent = Agent(
    client=chat_client,
    name="order_agent",
    description="Specialist for order status, delivery, and shipping questions.",
    instructions="Use tools for order and shipping facts. Explain the verified result concisely.",
    tools=[lookup_order, search_support_policy],
    require_per_service_call_history_persistence=True,
)

return_agent = Agent(
    client=chat_client,
    name="return_agent",
    description="Specialist for damaged products and return eligibility.",
    instructions=(
        "Verify the order and returns policy. Confirm customer intent before calling the simulated "
        "return tool. Hand off refund questions to the refund specialist."
    ),
    tools=[lookup_order, search_support_policy, create_return_request],
    require_per_service_call_history_persistence=True,
)

refund_agent = Agent(
    client=chat_client,
    name="refund_agent",
    description="Specialist for refund eligibility and simulated refund requests.",
    instructions=(
        "Verify the order and refunds policy. Confirm customer intent before calling the simulated "
        "refund tool. When everything is resolved, end with 'CASE CLOSED'."
    ),
    tools=[lookup_order, search_support_policy, create_refund_request],
    require_per_service_call_history_persistence=True,
)

## Build explicit handoff routes

The triage agent can reach every specialist. A return can lead to a refund. Specialists can return to triage when the customer's intent changes. Limiting the graph makes behavior easier to explain and evaluate than an unrestricted all-to-all mesh.

In [ ]:
handoff_builder = HandoffBuilder(
    name="customer_support_handoff",
    participants=[triage_agent, order_agent, return_agent, refund_agent],
    termination_condition=lambda conversation: (
        bool(conversation) and "case closed" in conversation[-1].text.lower()
    ),
).with_start_agent(triage_agent)

handoff_builder.add_handoff(triage_agent, [order_agent, return_agent, refund_agent])
handoff_builder.add_handoff(order_agent, [triage_agent])
handoff_builder.add_handoff(return_agent, [refund_agent, triage_agent])
handoff_builder.add_handoff(refund_agent, [triage_agent])

workflow_agent = handoff_builder.build().as_agent()
print("Workflow agent ready:", workflow_agent.name)

## Handle the interactive boundary

A handoff workflow pauses when an agent answers the customer instead of handing off. MAF represents that pause as a request for information. The helper below prints agent messages and collects pending requests so a UI—or this scripted demo—can respond.

In [ ]:
def display_and_collect_requests(
    response: AgentResponse,
) -> dict[str, HandoffAgentUserRequest]:
    pending: dict[str, HandoffAgentUserRequest] = {}

    for message in response.messages:
        if message.text:
            print(f"{message.author_name or message.role}: {message.text}")

        for content in message.contents:
            if content.type == "function_call" and content.name == WorkflowAgent.REQUEST_INFO_FUNCTION_NAME:
                request_args = WorkflowAgent.RequestInfoFunctionArgs.from_dict(content.arguments)
                pending[request_args.request_id] = request_args.request_event.data

    return pending

In [ ]:
scripted_customer_messages = [
    "Order ORD-1001 arrived damaged. I want to return it and receive a refund.",
    "Yes, create the simulated return and refund requests.",
    "Thanks for resolving this. Please close the case.",
]

initial_message = "Hello, I need help with a recent purchase."
print("customer:", initial_message)
workflow_response = await workflow_agent.run(initial_message)
pending_requests = display_and_collect_requests(workflow_response)

while pending_requests:
    if scripted_customer_messages:
        customer_message = scripted_customer_messages.pop(0)
        print("customer:", customer_message)
        responses = {
            request_id: HandoffAgentUserRequest.create_response(customer_message)
            for request_id in pending_requests
        }
    else:
        responses = {
            request_id: HandoffAgentUserRequest.terminate()
            for request_id in pending_requests
        }

    function_results = [
        Content("function_result", call_id=request_id, result=request_response)
        for request_id, request_response in responses.items()
    ]
    workflow_response = await workflow_agent.run(Message("tool", function_results))
    pending_requests = display_and_collect_requests(workflow_response)

## Discussion: when should you use handoffs?

**Good fit**

- Specialists need different instructions or tools.
- The next owner depends on conversational context.
- A specialist should speak directly with the user.

**Prefer a deterministic workflow when**

- Every step must happen in a fixed order.
- Compliance requires predictable approval gates.
- A normal function or rules engine can make the routing decision reliably.

Agents are not a replacement for authorization, transaction boundaries, or business invariants.

## From demo to production

1. Change consequential tools to require approval and enforce authorization inside each tool.
2. Replace in-memory workshop data with authenticated service clients.
3. Add trace correlation, latency metrics, and tool outcome telemetry.
4. Evaluate routing accuracy, task completion, policy adherence, and unnecessary handoffs.
5. Cap turns and provide an explicit human-escalation path.
6. Test every allowed handoff edge and failure mode before deployment.

In [ ]:
credential.close()

## Workshop recap

You progressed from secure Foundry connectivity to an application-owned agent, deterministic tools, structured output, sessions, a Foundry-managed agent, and a multi-agent handoff workflow. Continue with the optional advanced lessons to call a public MCP server, build an explicit executor-and-edge graph, and pause a workflow for human approval. The same MAF abstractions can move from notebooks into an API, background worker, or hosted agent.